# PCC_LEAKAGE_FREE_RERUN_2026 full run
GPU_REQUIRED five-fold, forty-case canonical execution.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
repo = Path('/kaggle/working/PCC')
subprocess.run(['git', 'clone', 'https://github.com/changxinjiresearch/PCC.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', '936a239d913a61533025c02918eeaf8b961d467f'], check=True)
artifact_root = repo / 'full_run_artifacts'; artifact_root.mkdir(parents=True, exist_ok=True)
canonical_root = Path('/kaggle/working/PCC_LEAKAGE_FREE_RERUN_2026')
if not canonical_root.exists(): canonical_root.symlink_to(artifact_root, target_is_directory=True)
subprocess.run([sys.executable, str(repo / 'experiments/run_pcc_leakage_free_full.py'), '--config', str(repo / 'configs/pcc_leakage_free_canonical.yaml'), '--gpu-required'], cwd=repo, check=True)
p0_cases = sorted((artifact_root / 'held_out_p0').glob('*/P0_COMPLETE.json'))
retros = sorted((artifact_root / 'retrospective/cases').glob('*/RETROSPECTIVE_COMPLETE.json'))
folds = sorted((artifact_root / 'folds').glob('fold_*/FOLD_COMPLETE.json'))
assert len(p0_cases) == 40 and len(retros) == 40 and len(folds) == 5
method_rows=[]; trajectory_rows=[]
for complete in retros:
    case = complete.parent; case_id = case.name
    with (case/'method_metrics.csv').open() as stream:
        method_rows.extend([{'case_id':case_id, **row} for row in csv.DictReader(stream)])
    with (case/'pcc_round_trajectory.csv').open() as stream:
        trajectory_rows.extend([{'case_id':case_id, **row} for row in csv.DictReader(stream)])
assert len(method_rows) == 280 and len(trajectory_rows) == 400
def write_csv(path, rows):
    with path.open('w', newline='') as stream:
        writer=csv.DictWriter(stream, fieldnames=tuple(rows[0])); writer.writeheader(); writer.writerows(rows)
write_csv(artifact_root/'ALL_CASE_METHOD_METRICS.csv', method_rows)
write_csv(artifact_root/'ALL_PCC_ROUND_TRAJECTORIES.csv', trajectory_rows)
(artifact_root/'FAILED_CASES.csv').write_text('case_id,stage,error\n')
summary={'status':'COMPLETE','gpu_required':True,'git_commit':'936a239d913a61533025c02918eeaf8b961d467f','config_sha256':'29111a4d9cb16a2981eec5cbaa193346f9715630a6e81aaf9734805f789a14b6','case_manifest_sha256':hashlib.sha256((artifact_root/'LOCKED_CASE_MANIFEST.csv').read_bytes()).hexdigest(),'fold_manifest_sha256':hashlib.sha256((artifact_root/'LOCKED_FOLD_MANIFEST.csv').read_bytes()).hexdigest(),'completed_folds':len(folds),'completed_p0_cases':len(p0_cases),'completed_retrospective_cases':len(retros),'method_rows':len(method_rows),'trajectory_rows':len(trajectory_rows),'failed_cases':0}
(artifact_root/'FULL_RUN_STATUS.json').write_text(json.dumps(summary, indent=2)); print(json.dumps(summary, indent=2))
